In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

In [ ]:
import sys, json, itertools, shutil, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import VGG19, InceptionV3, EfficientNetV2B0
from tensorflow.keras.applications import resnet_v2

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
try:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception as e:
    print("Memory growth not set (likely TF already initialized):", e)

In [ ]:
print(f"Num GPUs Available: {len(gpus)}")
for i, g in enumerate(gpus):
    print(f"GPU[{i}]: {g.name}")
print("Logical devices:", tf.config.list_logical_devices())

In [ ]:
TEST_DIR = "/kaggle/input/iqothnccd-lung-cancer-dataset/Test cases"
BASE_DIR = "/kaggle/input/iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset"
TRAIN_DIRS = {
    "Bengin": os.path.join(BASE_DIR, "Bengin cases"),
    "Malignant": os.path.join(BASE_DIR, "Malignant cases"),
    "Normal": os.path.join(BASE_DIR, "Normal cases"),
}

In [ ]:
WORK_ROOT = "./iqoth_merged"
if os.path.exists(WORK_ROOT):
    shutil.rmtree(WORK_ROOT)
for cls, p in TRAIN_DIRS.items():
    os.makedirs(os.path.join(WORK_ROOT, cls), exist_ok=True)
    for root, _, files in os.walk(p):
        for f in files:
            if f.lower().endswith((".png",".jpg",".jpeg",".bmp",".tif",".tiff")):
                src = os.path.join(root, f)
                dst = os.path.join(WORK_ROOT, cls, f"{abs(hash(src))}_{f}")
                try:
                    os.symlink(src, dst)
                except Exception:
                    shutil.copy2(src, dst)

class_names = sorted(list(TRAIN_DIRS.keys()))
num_classes = len(class_names)
print("Classes:", class_names)

In [ ]:
BATCH_SIZE = 16
VAL_SPLIT = 0.2
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS_HEAD = 10    
EPOCHS_FINE = 10    
LR_HEAD = 1e-3
LR_FINE = 1e-4

In [ ]:
def build_datasets(target_size, preprocess_fn):
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        WORK_ROOT,
        labels="inferred",
        label_mode="categorical",
        validation_split=VAL_SPLIT,
        subset="training",
        seed=SEED,
        image_size=target_size,
        batch_size=BATCH_SIZE,
        color_mode="rgb",
        shuffle=True,
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        WORK_ROOT,
        labels="inferred",
        label_mode="categorical",
        validation_split=VAL_SPLIT,
        subset="validation",
        seed=SEED,
        image_size=target_size,
        batch_size=BATCH_SIZE,
        color_mode="rgb",
        shuffle=False,
    )

    aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.05, 0.05),
    ], name="data_augmentation")

    def prep(x, y, training=False):
        x = tf.cast(x, tf.float32)
        x = preprocess_fn(x)
        if training:
            x = aug(x, training=True)
        return x, y

    train_ds = train_ds.map(lambda x,y: prep(x,y,True), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    val_ds   = val_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    return train_ds, val_ds

In [ ]:
def evaluate_model(model, ds, class_names, out_prefix):
    y_true, y_pred = [], []
    for x, y in ds:
        logits = model.predict(x, verbose=0)
        y_pred.append(np.argmax(logits, axis=1))
        y_true.append(np.argmax(y.numpy(), axis=1))
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Robust report: always pass labels spanning all classes
    report = classification_report(
        y_true, y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
    print(report)
    with open(f"{out_prefix}_report.txt", "w") as f:
        f.write(report)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    plt.figure(figsize=(5,4))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.ylabel("True"); plt.xlabel("Predicted"); plt.title("Confusion Matrix")
    plt.tight_layout(); plt.savefig(f"{out_prefix}_cm.png"); plt.close()

In [ ]:
def plot_history(history, out_png):
    hist = history.history
    plt.figure(figsize=(7,4))
    if "loss" in hist: plt.plot(hist["loss"], label="train_loss")
    if "val_loss" in hist: plt.plot(hist["val_loss"], label="val_loss")
    if "accuracy" in hist: plt.plot(hist["accuracy"], label="train_acc")
    if "val_accuracy" in hist: plt.plot(hist["val_accuracy"], label="val_acc")
    plt.xlabel("Epoch"); plt.ylabel("Value"); plt.title("Training curves"); plt.legend()
    plt.tight_layout(); plt.savefig(out_png); plt.close()

In [ ]:
def make_head(num_classes, dropout=0.3):
    return keras.Sequential([
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation="softmax")
    ], name="classifier_head")

In [ ]:
MODEL_SPECS = {
    "VGG19": (VGG19, tf.keras.applications.vgg19.preprocess_input, (224,224)),          # official sizes/preprocess [web:35][web:49]
    "ResNet101V2": (resnet_v2.ResNet101V2, tf.keras.applications.resnet_v2.preprocess_input, (224,224)),  # [web:37][web:35]
    "InceptionV3": (InceptionV3, tf.keras.applications.inception_v3.preprocess_input, (299,299)),         # [web:35]
    "EfficientNetV2B0": (EfficientNetV2B0, tf.keras.applications.efficientnet_v2.preprocess_input, (224,224)),  # [web:44][web:35]
}

In [ ]:
RESULTS = []
strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
print("Using strategy:", type(strategy).__name__)

In [ ]:
for model_name, (Constructor, preprocess_fn, size_xy) in MODEL_SPECS.items():
    print("\n" + "="*60)
    print(f"Training model: {model_name} | input {size_xy} | preprocess={preprocess_fn.__name__}")
    print("="*60)

    with strategy.scope():
        # Data inside scope for consistency on distributed setups [web:61]
        train_ds, val_ds = build_datasets(size_xy, preprocess_fn)

        # Build model
        base = Constructor(include_top=False, weights="imagenet", input_shape=(size_xy[0], size_xy[1], 3))
        base.trainable = False
        head = make_head(num_classes)

        inputs = keras.Input(shape=(size_xy[0], size_xy[1], 3))
        x = base(inputs, training=False)
        outputs = head(x)
        model = keras.Model(inputs, outputs, name=f"{model_name}_clf")

        # Callbacks
        out_prefix = f"./out_{model_name}"
        ckpt = keras.callbacks.ModelCheckpoint(f"{out_prefix}_best.h5", monitor="val_accuracy",
                                               save_best_only=True, save_weights_only=False, mode="max")
        es = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, mode="max")
        rlrop = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

        # Compile and head training
        model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])
        print(model.summary())
        history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=[ckpt, es, rlrop])

        # Fine-tune last 20% of backbone
        n_layers = len(base.layers)
        unfreeze_from = int(n_layers * 0.8)
        for i, lyr in enumerate(base.layers):
            lyr.trainable = (i >= unfreeze_from)
        model.compile(optimizer=keras.optimizers.Adam(LR_FINE),
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])
        hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=[ckpt, es, rlrop])

        # Merge histories for plotting
        for k, v in hist2.history.items():
            history.history[k] = history.history.get(k, []) + v

        # Curves and evaluation
        plot_history(history, f"{out_prefix}_training_curves.png")
        print(f"Evaluating {model_name} ...")
        evaluate_model(model, val_ds, class_names, out_prefix)
        val_loss, val_acc = model.evaluate(val_ds, verbose=0)
        RESULTS.append({"model": model_name, "val_accuracy": float(val_acc), "val_loss": float(val_loss)})

In [1]:
res_df = pd.DataFrame(RESULTS).sort_values("val_accuracy", ascending=False)
print("\nModel performance summary:")
print(res_df.to_string(index=False))
res_df.to_csv("./model_performance_summary.csv", index=False)

plt.figure(figsize=(7,4))
sns.barplot(data=res_df, x="model", y="val_accuracy", color="steelblue")
plt.ylim(0, 1.0)
plt.title("Validation Accuracy by Model")
plt.ylabel("Val Accuracy")
plt.tight_layout()
plt.savefig("./model_val_accuracy_bar.png")
plt.close()

2025-10-25 17:06:59.774086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761412019.796217    2636 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761412019.803045    2636 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Num GPUs Available: 2
GPU[0]: /physical_device:GPU:0
GPU[1]: /physical_device:GPU:1
Logical devices: [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU'), LogicalDevice(name='/device:GPU:1', device_type='GPU')]


I0000 00:00:1761412022.652478    2636 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761412022.653172    2636 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Classes: ['Bengin', 'Malignant', 'Normal']
Using strategy: MirroredStrategy

Training model: VGG19 | input (224, 224) | preprocess=preprocess_input
Found 1097 files belonging to 3 classes.
Using 878 files for training.
Found 1097 files belonging to 3 classes.
Using 219 files for validation.


Model: "VGG19_clf"

...

None
Epoch 1/10


E0000 00:00:1761412714.935753    2636 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetV2B0_clf_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


55/55 ━━━━━━━━━━━━━━━━━━━━ 34s 294ms/step - accuracy: 0.4210 - loss: 1.0535 - val_accuracy: 1.0000 - val_loss: 0.0752 - learning_rate: 0.0010
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 128ms/step - accuracy: 0.6900 - loss: 0.7918 - val_accuracy: 1.0000 - val_loss: 0.0937 - learning_rate: 0.0010
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 127ms/step - accuracy: 0.8753 - loss: 0.3884 - val_accuracy: 1.0000 - val_loss: 0.0859 - learning_rate: 0.0010
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 129ms/step - accuracy: 0.8442 - loss: 0.4347 - val_accuracy: 1.0000 - val_loss: 0.0948 - learning_rate: 5.0000e-04
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 127ms/step - accuracy: 0.8156 - loss: 0.4161 - val_accuracy: 1.0000 - val_loss: 0.1206 - learning_rate: 5.0000e-04
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 131ms/step - accuracy: 0.9249 - loss: 0.2329 - val_accuracy: 0.9919 - val_loss: 0.1677 - learning_rate: 2.5000e-04
Epoch 1/10


E0000 00:00:1761412796.748033    2636 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetV2B0_clf_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


55/55 ━━━━━━━━━━━━━━━━━━━━ 42s 313ms/step - accuracy: 0.7766 - loss: 0.6104 - val_accuracy: 0.9922 - val_loss: 0.2121 - learning_rate: 1.0000e-04
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.8330 - loss: 0.3986 - val_accuracy: 0.9922 - val_loss: 0.1670 - learning_rate: 1.0000e-04
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 145ms/step - accuracy: 0.8425 - loss: 0.3444 - val_accuracy: 0.9919 - val_loss: 0.2330 - learning_rate: 1.0000e-04
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.8444 - loss: 0.4081 - val_accuracy: 0.9919 - val_loss: 0.1925 - learning_rate: 1.0000e-04
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 152ms/step - accuracy: 0.8854 - loss: 0.3257 - val_accuracy: 0.9919 - val_loss: 0.1821 - learning_rate: 5.0000e-05
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 9s 152ms/step - accuracy: 0.8466 - loss: 0.3234 - val_accuracy: 0.9899 - val_loss: 0.1979 - learning_rate: 5.0000e-05
Evaluating EfficientNetV2B0 ...
              precision    recall  f1-scor